# Notebook 02: Model Exploration

Pure exploration -- no saved outputs except findings documented in PROGRESS.md. Run this notebook BEFORE writing any embedding generation code. Verify all API details that src/embeddings/ stubs depend on.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys

REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
REPO_DIR = '/content/antibody-property-prediction'
BRANCH = 'implementation'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready.")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project/Antibody_Project')
# DATA_DIR is in the repo (data/ at repo root) -- comes from src.config
EMBEDDING_DIR = DRIVE_ROOT / 'embeddings'
RESULTS_DIR = DRIVE_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths set.")

In [ ]:
!apt-get install -y hmmer
!pip install -q fair-esm ablang2 anarci wandb

In [ ]:
!pip install -q --upgrade ipython

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
subprocess.run(['find', '/content/antibody-property-prediction', '-type', 'd',
                '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
               capture_output=True)
print("Autoreload enabled, pycache cleared.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import wandb
wandb.login()

## Imports

Loads the two embedding models and the data loaders. `DEVICE` resolves to `mps` on Apple Silicon (Metal GPU), `cuda` on a Colab GPU, or `cpu` as fallback -- set in `src/config.py`.

Both embedding modules (`src/embeddings/esm2`, `src/embeddings/ablang2`) expose the same interface pattern: `load_*`, `embed_batch`, `profile_throughput`. The `get_*` helpers expose internal model constants (repr layer, special token ids, hidden dim) without hardcoding them.

In [2]:
import json
import numpy as np
import pandas as pd
import torch

from src.config import DATA_DIR, DEVICE, ESM2_MODEL_NAME
from src.data.abagym import load_abagym_antibody, load_abagym_sequences, get_mutation_site_index
from src.embeddings.esm2 import (
    load_esm2, get_repr_layer, get_token_offsets,
    embed_batch as esm2_embed_batch,
    profile_throughput as esm2_profile,
)
from src.embeddings.ablang2 import (
    load_ablang2, get_special_tokens, get_hidden_dim, get_chain_masks,
    embed_batch as ablang2_embed_batch,
    profile_throughput as ablang2_profile,
)

print(f"Device: {DEVICE}")

Device: mps


## ESM-2: Load Model

ESM-2 (Evolutionary Scale Modeling 2) is a protein language model from Meta trained on 250M UniRef50 sequences. We use the `esm2_t33_650M_UR50D` variant: 33 transformer layers, 651M parameters, trained on the UR50D (UniRef50 clustered at 50% identity) dataset.

The model is loaded via `esm.pretrained`, which downloads weights on first use and caches them at `~/.cache/torch/hub/checkpoints/`. The `alphabet` object handles tokenization; `batch_converter` converts `(label, sequence)` pairs into padded token tensors.

`repr_layer = model.num_layers` (= 33) selects the final transformer layer as the embedding source. This is read from the model object rather than hardcoded so the code works unchanged if a different ESM-2 variant is substituted.

In [3]:
# ESM-2 can have MPS compatibility issues -- if this fails, change DEVICE to 'cpu'
print(f"Loading ESM-2 ({ESM2_MODEL_NAME}) on {DEVICE}...")
model, alphabet, batch_converter = load_esm2(DEVICE)
repr_layer = get_repr_layer(model)

print(f"Repr layer: {repr_layer}  (= model.num_layers, never hardcode)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading ESM-2 (esm2_t33_650M_UR50D) on mps...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /Users/oscarrodriguez/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /Users/oscarrodriguez/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Repr layer: 33  (= model.num_layers, never hardcode)
Parameters: 651,043,254


## ESM-2: Inspect Tokenizer and Special Tokens

Before extracting residue-level embeddings, we need to know exactly how ESM-2 encodes a sequence as a token array.

**Special tokens** are non-amino-acid tokens the model uses for bookkeeping:
- **BOS (Beginning of Sequence)** / `cls_idx` (0): always inserted at position 0.
- **EOS (End of Sequence)** / `eos_idx` (2): appended at the last position.
- **PAD (Padding)** / `padding_idx` (1): fills shorter sequences in a batch to uniform length. Ignored during attention.
- **UNK (Unknown)** (3): assigned if a character is not recognized. If this appears during embedding generation, the embedding at that position is uninformative.
- **MASK** (32): masking token from ESM-2's pretraining. Never appears in our forward passes.

**Full vocabulary** (32 entries, confirmed by inspecting `alphabet.tok_to_idx`): indices 4-23 are the 20 standard amino acids, 24-28 are ambiguous/non-standard AAs (X, B, U, Z, O), 29-30 are MSA gap characters (`.`, `-`), and 31 is a reserved placeholder `<null_1>`. None of 24-31 are expected in AbAgym sequences.

**Why this matters for residue extraction:** BOS at position 0 shifts every amino acid right by one -- residue at sequence index `i` maps to token position `i + 1` (`residue_offset = 1`).

Confirmed: BOS=0, EOS=2, PAD=1. For `EVQL`: token array is `[0, E, V, Q, L, 2]` -- 6 tokens for 4 residues.

In [4]:
offsets = get_token_offsets(alphabet)
print("Special token indices:")
for k, v in offsets.items():
    print(f"  {k}: {v}")

# Show token structure on a short example
_, _, tokens = batch_converter([("test", "EVQL")])
print(f"\nTokens for 'EVQL': {tokens[0].tolist()}")
print(f"  pos 0 = {tokens[0,0].item()} (BOS = cls_idx)")
print(f"  pos 1-4 = residues")
print(f"  pos 5 = {tokens[0,5].item()} (EOS = eos_idx)")
print(f"\nresidual_offset = {offsets['residue_offset']}: seq_idx + 1 = token position")

Special token indices:
  bos_idx: 0
  eos_idx: 2
  pad_idx: 1
  residue_offset: 1

Tokens for 'EVQL': [0, 9, 7, 16, 4, 2]
  pos 0 = 0 (BOS = cls_idx)
  pos 1-4 = residues
  pos 5 = 2 (EOS = eos_idx)

residual_offset = 1: seq_idx + 1 = token position


In [7]:
print("Full ESM-2 vocabulary:")
for tok, idx in sorted(alphabet.tok_to_idx.items(), key=lambda x: x[1]):
    print(f"  {idx:3d}: {repr(tok)}")

Full ESM-2 vocabulary:
    0: '<cls>'
    1: '<pad>'
    2: '<eos>'
    3: '<unk>'
    4: 'L'
    5: 'A'
    6: 'G'
    7: 'V'
    8: 'S'
    9: 'E'
   10: 'R'
   11: 'T'
   12: 'I'
   13: 'D'
   14: 'P'
   15: 'K'
   16: 'Q'
   17: 'N'
   18: 'F'
   19: 'Y'
   20: 'M'
   21: 'H'
   22: 'W'
   23: 'C'
   24: 'X'
   25: 'B'
   26: 'U'
   27: 'Z'
   28: 'O'
   29: '.'
   30: '-'
   31: '<null_1>'
   32: '<mask>'


## ESM-2: Test Sequence Forward Pass

Runs a single heavy+light pair through ESM-2 to confirm output shapes before writing any embedding generation code.

ESM-2 processes each chain independently -- heavy and light are passed as separate forward calls. The output tensor for a sequence of length `L` has shape `(1, L+2, 1280)`: batch size 1, `L+2` positions (one BOS at index 0, `L` residues, one EOS at index `L+1`), and 1280-dimensional embeddings at each position.

The two chains are later concatenated along the embedding dimension to produce a 2560-dim sequence-level representation `[H_pool || L_pool]`.

Confirmed on Ang2_2017_G6 (H=215, L=213): heavy repr `(1, 217, 1280)`, light repr `(1, 215, 1280)`.

In [5]:
sequences_df = load_abagym_sequences(DATA_DIR)
ang2 = sequences_df[sequences_df['dms_name'] == 'Ang2_2017_G6'].iloc[0]
heavy_seq = ang2['heavy_seq']
light_seq = ang2['light_seq']

print(f"Ang2_2017_G6: H={len(heavy_seq)} residues, L={len(light_seq)} residues")

with torch.no_grad():
    _, _, h_tokens = batch_converter([("H", heavy_seq)])
    h_tokens = h_tokens.to(DEVICE)
    h_out  = model(h_tokens, repr_layers=[repr_layer], return_contacts=False)
    h_repr = h_out['representations'][repr_layer]  # (1, len(H)+2, 1280)

    _, _, l_tokens = batch_converter([("L", light_seq)])
    l_tokens = l_tokens.to(DEVICE)
    l_out  = model(l_tokens, repr_layers=[repr_layer], return_contacts=False)
    l_repr = l_out['representations'][repr_layer]  # (1, len(L)+2, 1280)

print(f"\nHeavy repr shape: {tuple(h_repr.shape)}  -- (batch, len(H)+2, 1280)")
print(f"Light repr shape: {tuple(l_repr.shape)}  -- (batch, len(L)+2, 1280)")
print(f"  +2 accounts for BOS at position 0 and EOS at position len+1")

Ang2_2017_G6: H=215 residues, L=213 residues

Heavy repr shape: (1, 217, 1280)  -- (batch, len(H)+2, 1280)
Light repr shape: (1, 215, 1280)  -- (batch, len(L)+2, 1280)
  +2 accounts for BOS at position 0 and EOS at position len+1


## ESM-2: Verify Residue Indexing

Confirms that the BOS offset (`token_pos = seq_idx + 1`) correctly maps a known mutation site to the right token position.

We use the first mutation in Ang2_2017_G6 (H:P100A) as a ground-truth check. `get_mutation_site_index` returns the 0-based index into the amino acid string from ANARCI's IMGT mapping. Adding 1 gives the token position in ESM-2's token array.

**Verified on Ang2_2017_G6 H:P100A:**
- `seq_idx = 103` (0-based index into the 215-residue heavy chain string)
- `token_pos = 104` (= seq_idx + 1 for BOS)
- AA at seq_idx 103: `'P'` -- matches wildtype, confirming the index is correct
- Residue embedding: shape `(1280,)`, norm = 10.13
- BOS embedding norm = 10.03 -- differs from residue embedding, confirming distinct representations

Note: BOS and residue embedding norms are similar in magnitude, which is normal for ESM-2. The L2 norm alone does not distinguish token types -- the 1280-dimensional direction is what encodes residue identity.

In [8]:
antibody_df = load_abagym_antibody(DATA_DIR)

# First mutation in Ang2_2017_G6
row     = antibody_df[antibody_df['DMS_name'] == 'Ang2_2017_G6'].iloc[0]
dms_name = row['DMS_name']
chain    = row['chains']
site     = row['site']
wt_aa    = row['wildtype']
mut_aa   = row['mutation']

seq_idx   = get_mutation_site_index(sequences_df, dms_name, chain, site)
token_pos = seq_idx + 1  # BOS offset

print(f"Mutation:  {dms_name} {chain}:{wt_aa}{site}{mut_aa}")
print(f"seq_idx:   {seq_idx}  (0-based index into amino acid string)")
print(f"token_pos: {token_pos} (seq_idx + 1 for BOS)")

# Confirm the AA at seq_idx matches the wildtype
seq = heavy_seq if chain == 'H' else light_seq
assert seq[seq_idx] == wt_aa, f"Mismatch: got {seq[seq_idx]}, expected {wt_aa}"
print(f"\nAA at seq_idx {seq_idx}: '{seq[seq_idx]}' -- matches wildtype '{wt_aa}'")

# Extract the residue embedding
with torch.no_grad():
    _, _, tokens = batch_converter([(chain, seq)])
    tokens = tokens.to(DEVICE)
    out = model(tokens, repr_layers=[repr_layer], return_contacts=False)
    repr_full = out['representations'][repr_layer]

residue_emb = repr_full[0, token_pos, :].cpu()
bos_emb     = repr_full[0, 0, :].cpu()

print(f"\nResidue embedding at token_pos {token_pos}: shape={tuple(residue_emb.shape)}, norm={residue_emb.norm().item():.4f}")
print(f"BOS embedding at token_pos 0:              norm={bos_emb.norm().item():.4f}  (should differ)")

Mutation:  Ang2_2017_G6 H:P100A
seq_idx:   103  (0-based index into amino acid string)
token_pos: 104 (seq_idx + 1 for BOS)

AA at seq_idx 103: 'P' -- matches wildtype 'P'

Residue embedding at token_pos 104: shape=(1280,), norm=10.1285
BOS embedding at token_pos 0:              norm=10.0261  (should differ)


## ESM-2: Throughput Profiling

Profiles forward pass speed across sequence lengths [50, 100, 150, 200, 250] and batch sizes [1, 8, 32, 64] to inform batch size selection for embedding generation. VRAM is reported as `nan` on MPS (Apple Silicon) since peak memory tracking requires CUDA; this column is meaningful on Colab GPU.

**Results on MPS (Apple M-series):**

Key observations:
- Batching helps substantially up to bs=32. At length 50, bs=1 → bs=32 is ~40x faster per sample (0.51s → 0.016s).
- At longer sequences (150+), gains plateau and slightly reverse above bs=32, consistent with memory bandwidth saturation on MPS unified memory.
- Antibody chains in AbAgym are ~200-215 residues. At length=200, bs=32: **0.054 sec/sample**.
- Estimated cost for 5318 residue-level embeddings at bs=32: ~5 minutes on MPS. Colab CUDA will be significantly faster.

**Implication for NB03:** Use bs=32 as the default batch size for ESM-2 embedding generation. Monitor for MPS OOM at larger batch sizes on longer sequences.

In [9]:
print(f"ESM-2 throughput profile on {DEVICE}:")
print()
esm2_profile(model, alphabet, batch_converter, DEVICE)

ESM-2 throughput profile on mps:

  length  batch   sec/sample   peak VRAM GB
----------------------------------------------
      50      1       0.5061            nan
      50      8       0.1220            nan
      50     32       0.0159            nan
      50     64       0.0127            nan
     100      1       0.2655            nan
     100      8       0.0516            nan
     100     32       0.0250            nan
     100     64       0.0271            nan
     150      1       0.1902            nan
     150      8       0.0484            nan
     150     32       0.0370            nan
     150     64       0.0421            nan
     200      1       0.3160            nan
     200      8       0.0660            nan
     200     32       0.0541            nan
     200     64       0.0673            nan
     250      1       0.1566            nan
     250      8       0.0702            nan
     250     32       0.0762            nan
     250     64       0.0834           

## AbLang2: Load Model

AbLang2-paired is an antibody-specific language model from the Oxford Protein Informatics Group, trained on the Observed Antibody Space (OAS) database via masked language modeling. Unlike ESM-2 which processes arbitrary protein sequences, AbLang2-paired is designed specifically for paired VH+VL antibody inputs.

The model downloads as a `.tar` archive on first use; the `x` lines during download are tar extraction output (not errors). Weights are cached locally after the first download.

`ablang.AbRep` is the representation module (analogous to ESM-2's transformer body). `ablang.tokenizer` handles sequence encoding. There is no repr layer selection -- AbLang2 always returns the final layer's hidden states.

**Confirmed on load:**
- SEP token id: 25 (`ablang.tokenizer.sep_token`) -- separates heavy and light chains in the token sequence
- PAD token id: 21 (`ablang.AbRep.padding_tkn`) -- fills shorter sequences in a batch
- Hidden dim: 480 (NOT 768 -- the original spec assumed 768, confirmed empirically to be 480)
- Parameters: 44,348,304 (~44M) -- ~15x smaller than ESM-2 (651M)
- Combined memory footprint: ESM-2 (~2.5 GB) + AbLang2 (~0.17 GB) -- both fit simultaneously on MPS

In [10]:
print(f"Loading AbLang2-paired on {DEVICE}...")
ablang = load_ablang2(DEVICE)
print("Loaded.")

special    = get_special_tokens(ablang)
hidden_dim = get_hidden_dim(ablang, DEVICE)

print(f"\nSEP token id: {special['sep']}  (= ablang.tokenizer.sep_token)")
print(f"PAD token id: {special['pad']}  (= ablang.AbRep.padding_tkn)")
print(f"Hidden dim:   {hidden_dim}  (expected 480)")

n_params = sum(p.numel() for p in ablang.AbRep.parameters())
print(f"Parameters:   {n_params:,}")

assert hidden_dim == 480, f"Unexpected hidden dim: {hidden_dim}"
print("Assertion passed.")

Loading AbLang2-paired on mps...


x hparams.json
x model.pt


Loaded.

SEP token id: 25  (= ablang.tokenizer.sep_token)
PAD token id: 21  (= ablang.AbRep.padding_tkn)
Hidden dim:   480  (expected 480)
Parameters:   44,348,304
Assertion passed.


## AbLang2: Inspect Tokenizer and Special Tokens

AbLang2's tokenization scheme is fundamentally different from ESM-2's. The heavy and light chains are concatenated into a single string separated by `|`, which the tokenizer converts to a SEP token (id=25). With `w_extra_tkns=False` (always used here), no start or end tokens are added -- the sequence starts immediately with the first heavy chain residue at position 0.

**Full vocabulary** (26 entries, confirmed via `ablang.tokenizer.token_to_aa`): indices 1-20 are the 20 standard amino acids. Special tokens: start=0 (`<`), end=22 (`>`), pad=21 (`-`), sep=25 (`|`), mask=23 (`*`), unknown=24 (`X`). With `w_extra_tkns=False`, start (0) and end (22) are suppressed from output. If `w_extra_tkns=True` were used, start would appear at position 0 and shift all residues right by 1 -- matching ESM-2's layout. We always use `False`.

**Token layout (`w_extra_tkns=False`):**
```
[H_1, H_2, ..., H_n, SEP(25), L_1, L_2, ..., L_m, PAD(21), ...]
```

**Residue position rules:**
- Heavy residue at sequence index `i`: token position = `i` (no offset)
- SEP: token position = `len(heavy_seq)`
- Light residue at sequence index `j`: token position = `len(heavy_seq) + 1 + j`

**Confirmed on `EVQL|DIQM`:** token array `[E, V, Q, L, SEP(25), D, I, Q, M]`, SEP at position 4 = `len('EVQL')`.

In [11]:
test_seq = "EVQL|DIQM"
tokens_test = ablang.tokenizer([test_seq], pad=True, w_extra_tkns=False, device=str(DEVICE))

print(f"Tokens for '{test_seq}': {tokens_test[0].tolist()}")
print(f"\nNo BOS or EOS tokens -- contrast with ESM-2")
print(f"  Heavy residues: positions 0 to {len('EVQL') - 1}")
print(f"  SEP at position: {len('EVQL')}  (= len(heavy_seq))")
print(f"  Light residues: positions {len('EVQL') + 1} to {len('EVQL') + len('DIQM')}")

sep_pos = (tokens_test[0] == special['sep']).nonzero().flatten().tolist()
print(f"\nActual SEP position: {sep_pos}  (expected: [{len('EVQL')}])")

Tokens for 'EVQL|DIQM': [6, 15, 10, 20, 25, 5, 16, 10, 1]

No BOS or EOS tokens -- contrast with ESM-2
  Heavy residues: positions 0 to 3
  SEP at position: 4  (= len(heavy_seq))
  Light residues: positions 5 to 8

Actual SEP position: [4]  (expected: [4])


In [14]:
print("Full AbLang2 vocabulary (token_to_aa):")
for idx, tok in sorted(ablang.tokenizer.token_to_aa.items()):
    print(f"  {idx:3d}: {repr(tok)}")

print(f"\nSpecial tokens:")
print(f"  start:   {ablang.tokenizer.start_token}")
print(f"  end:     {ablang.tokenizer.end_token}")
print(f"  pad:     {ablang.tokenizer.pad_token}")
print(f"  sep:     {ablang.tokenizer.sep_token}")
print(f"  mask:    {ablang.tokenizer.mask_token}")
print(f"  unknown: {ablang.tokenizer.unknown_token}")

Full AbLang2 vocabulary (token_to_aa):
    0: '<'
    1: 'M'
    2: 'R'
    3: 'H'
    4: 'K'
    5: 'D'
    6: 'E'
    7: 'S'
    8: 'T'
    9: 'N'
   10: 'Q'
   11: 'C'
   12: 'G'
   13: 'P'
   14: 'A'
   15: 'V'
   16: 'I'
   17: 'F'
   18: 'Y'
   19: 'W'
   20: 'L'
   21: '-'
   22: '>'
   23: '*'
   24: 'X'
   25: '|'

Special tokens:
  start:   0
  end:     22
  pad:     21
  sep:     25
  mask:    23
  unknown: 24


## AbLang2: Test Sequence Forward Pass

Runs a single VH|VL input through AbLang2 to confirm output shapes. Unlike ESM-2, both chains are embedded in a single forward pass -- the full concatenated sequence `[H_1...H_n, SEP, L_1...L_m]` is processed together, allowing cross-chain attention.

Output shape is `(batch, seq_len, 480)` where `seq_len = len(heavy) + 1 (SEP) + len(light)`. No +2 for BOS/EOS (suppressed by `w_extra_tkns=False`).

**Confirmed on Ang2_2017_G6 (H=215, L=213):**
- Expected seq_len: 215 + 1 + 213 = 429
- Output repr shape: `(1, 429, 480)` -- exact match
- Hidden dim: 480 (confirmed)

In [16]:
full_seq = f"{heavy_seq}|{light_seq}"
tokens_ang2 = ablang.tokenizer([full_seq], pad=True, w_extra_tkns=False, device=str(DEVICE))

with torch.no_grad():
    repr_ablang_full = ablang.AbRep(tokens_ang2).last_hidden_states  # (1, len(H)+1+len(L), 480)

print(f"Ang2_2017_G6 AbLang2 forward pass:")
print(f"  Heavy seq length: {len(heavy_seq)}")
print(f"  Light seq length: {len(light_seq)}")
print(f"  Expected seq_len: {len(heavy_seq)} + 1 (SEP) + {len(light_seq)} = {len(heavy_seq) + 1 + len(light_seq)}")
print(f"  Actual token seq_len: {tokens_ang2.shape[1]}")
print(f"  Output repr shape:    {tuple(repr_ablang_full.shape)}  -- (batch, seq_len, 480)")
print(f"  Hidden dim = {repr_ablang_full.shape[-1]}  (expected 480)")

Ang2_2017_G6 AbLang2 forward pass:
  Heavy seq length: 215
  Light seq length: 213
  Expected seq_len: 215 + 1 (SEP) + 213 = 429
  Actual token seq_len: 429
  Output repr shape:    (1, 429, 480)  -- (batch, seq_len, 480)
  Hidden dim = 480  (expected 480)


## AbLang2: Verify Chain Boundary Detection

Since both chains are embedded in a single forward pass, we need to reliably separate heavy and light residue tokens when extracting per-residue embeddings or computing chain-specific mean pools. `get_chain_masks` does this by locating the SEP token and constructing boolean masks.

The heavy mask covers all positions before SEP; the light mask covers all positions after SEP, excluding PAD tokens. Both masks exclude the SEP position itself.

**Confirmed on Ang2_2017_G6:**
- SEP at position 215 = `len(heavy_seq)` -- correct
- Tokens masked as heavy: 215 (expected 215)
- Tokens masked as light: 213 (expected 213)
- Assertions passed -- masks account for all residues with no overlap or leakage

In [17]:
heavy_mask, light_mask = get_chain_masks(tokens_ang2, special['sep'], special['pad'])

h_count = heavy_mask[0].sum().item()
l_count = light_mask[0].sum().item()

print(f"Ang2_2017_G6 chain boundary detection:")
print(f"  Heavy seq length:        {len(heavy_seq)}")
print(f"  Light seq length:        {len(light_seq)}")
print(f"  Tokens masked as heavy:  {int(h_count)}  (expected: {len(heavy_seq)})")
print(f"  Tokens masked as light:  {int(l_count)}  (expected: {len(light_seq)})")
print(f"  SEP position:            {(tokens_ang2[0] == special['sep']).nonzero().flatten().tolist()}  (expected: [{len(heavy_seq)}])")

assert int(h_count) == len(heavy_seq), f"Heavy mask mismatch: got {int(h_count)}, expected {len(heavy_seq)}"
assert int(l_count) == len(light_seq), f"Light mask mismatch: got {int(l_count)}, expected {len(light_seq)}"
print("Assertions passed.")

Ang2_2017_G6 chain boundary detection:
  Heavy seq length:        215
  Light seq length:        213
  Tokens masked as heavy:  215  (expected: 215)
  Tokens masked as light:  213  (expected: 213)
  SEP position:            [215]  (expected: [215])
Assertions passed.


## AbLang2: Verify Residue Indexing

Confirms that the chain-specific token position rules correctly map a known mutation site to the right token. Uses the same mutation as the ESM-2 verification (Ang2_2017_G6 H:P100A) for direct comparison.

For the heavy chain, `token_pos = seq_idx` with no offset -- residues start at position 0 with no BOS. For the light chain, `token_pos = len(heavy_seq) + 1 + seq_idx` to skip past all heavy tokens and the SEP.

**Verified on Ang2_2017_G6 H:P100A:**
- `seq_idx = 103`, `token_pos = 103` (heavy chain, no offset)
- Token id at position 103: 13 = `'P'` -- matches wildtype, correct residue targeted
- Not SEP (25), not PAD (21) -- confirmed clean residue token
- Residue embedding: shape `(480,)`, norm = 7.25

**Comparison with ESM-2:** same mutation, same seq_idx (103), but token_pos differs: ESM-2 gives 104 (seq_idx + 1 for BOS), AbLang2 gives 103 (seq_idx directly). This offset difference must be handled correctly in `embed_sequences_residue` for each model.

In [18]:
# Same mutation as ESM-2 verification: dms_name, chain, site, wt_aa, seq_idx from above
if chain == 'H':
    token_pos_ablang = seq_idx
else:
    token_pos_ablang = len(heavy_seq) + 1 + seq_idx  # +1 for SEP

print(f"Mutation:  {dms_name} {chain}:{wt_aa}{site}{mut_aa}")
print(f"seq_idx:   {seq_idx}")
print(f"token_pos: {token_pos_ablang}")
print(f"  Heavy: token_pos = seq_idx (no BOS offset)")
print(f"  Light: token_pos = len(heavy) + 1 + seq_idx (+1 for SEP)")

# Confirm the token at this position is not a special token
token_at_pos = tokens_ang2[0, token_pos_ablang].item()
print(f"\nToken id at position {token_pos_ablang}: {token_at_pos}")
print(f"  Is SEP: {token_at_pos == special['sep']}  (expected: False)")
print(f"  Is PAD: {token_at_pos == special['pad']}  (expected: False)")
assert token_at_pos != special['sep'] and token_at_pos != special['pad']

# Extract residue embedding
with torch.no_grad():
    repr_ablang = ablang.AbRep(tokens_ang2).last_hidden_states  # (1, seq_len, 480)

residue_emb_ablang = repr_ablang[0, token_pos_ablang, :].cpu()
print(f"\nResidue embedding shape: {tuple(residue_emb_ablang.shape)}  (expected: (480,))")
print(f"  Norm: {residue_emb_ablang.norm().item():.4f}")

Mutation:  Ang2_2017_G6 H:P100A
seq_idx:   103
token_pos: 103
  Heavy: token_pos = seq_idx (no BOS offset)
  Light: token_pos = len(heavy) + 1 + seq_idx (+1 for SEP)

Token id at position 103: 13
  Is SEP: False  (expected: False)
  Is PAD: False  (expected: False)

Residue embedding shape: (480,)  (expected: (480,))
  Norm: 7.2549


## AbLang2: Throughput Profiling

Profiles forward pass speed across sequence lengths [50, 100, 150, 200, 250] and batch sizes [1, 8, 32, 64]. Same profiling setup as ESM-2 for direct comparison. VRAM reports `nan` on MPS.

**Results on MPS (Apple M-series):**

AbLang2 is significantly faster than ESM-2 across all conditions -- ~10x at bs=1, ~2x at bs=32 for length=200. Consistent with the 15x parameter count difference (44M vs 651M).

Key observations:
- First row (length=50, bs=1: 0.524 sec) is anomalously slow due to MPS JIT compilation on the first call -- not representative of steady-state performance.
- Batching gains plateau earlier than ESM-2. At length=200, bs=8 is fastest (0.019 sec/sample); bs=32 and bs=64 are slightly slower, suggesting MPS saturation occurs at smaller batch sizes for this lighter model.
- AbLang2 processes paired H+L sequences in practice (~429 tokens for Ang2_2017_G6), longer than the profiled lengths. Real-world per-sample cost is closer to the 250-row estimates.
- At length=250, bs=32: **0.025 sec/sample**. Estimated cost for 5318 embeddings: ~2-3 minutes on MPS.

**Implication for NB03:** Use bs=32 for AbLang2 embedding generation (consistent with ESM-2). Monitor first-batch warmup time on Colab -- discard it from throughput estimates.

In [20]:
print(f"AbLang2 throughput profile on {DEVICE}:")
print()
ablang2_profile(ablang, DEVICE)

AbLang2 throughput profile on mps:

  length  batch   sec/sample   peak VRAM GB
----------------------------------------------
      50      1       0.5242            nan
      50      8       0.0507            nan
      50     32       0.0152            nan
      50     64       0.0035            nan
     100      1       0.0201            nan
     100      8       0.0087            nan
     100     32       0.0189            nan
     100     64       0.0130            nan
     150      1       0.0262            nan
     150      8       0.0130            nan
     150     32       0.0121            nan
     150     64       0.0264            nan
     200      1       0.0278            nan
     200      8       0.0185            nan
     200     32       0.0287            nan
     200     64       0.0321            nan
     250      1       0.0304            nan
     250      8       0.0272            nan
     250     32       0.0246            nan
     250     64       0.0427         

## Findings Summary

All API details confirmed and documented in PROGRESS.md (Notebook 02 section). No src/ changes required -- all stubs were written with correct logic prior to this notebook.

Key findings:
- ESM-2: BOS offset applies (`token_pos = seq_idx + 1`), repr_layer=33, hidden_dim=1280, separate H/L passes
- AbLang2: no BOS offset for heavy (`token_pos = seq_idx`), light uses `len(heavy)+1+seq_idx`, hidden_dim=480 (not 768), single paired pass
- Both models verified on same mutation (Ang2_2017_G6 H:P100A, seq_idx=103) -- token positions differ by exactly 1 due to BOS
- Recommended batch size for NB03: bs=32 for both models